# Proyecto 2 (alternativa sin instalar nada): SQL con DuckDB

Las mismas consultas de `analisis_sql.sql`, pero corriendo con **DuckDB**: un motor SQL que funciona dentro de Python, sin servidor ni base de datos que instalar. Sirve en Google Colab y en tu compu.

Los resultados de *ventas por mes* tienen que coincidir con los del notebook de Pandas.

In [ ]:
%pip install -q duckdb

In [ ]:
import os
import duckdb

# Si ventas.csv está en la misma carpeta (corriendo local) lo usamos;
# si no (por ejemplo en Google Colab), lo bajamos directo de GitHub.
URL_CSV = "https://raw.githubusercontent.com/marianacasella92/masterclass-data-analytics/main/ventas.csv"
ruta = "ventas.csv" if os.path.exists("ventas.csv") else URL_CSV

# Creamos la tabla ventas. La fecha viene como día/mes/año, así que se lo indicamos.
duckdb.sql(f"""
    CREATE OR REPLACE TABLE ventas AS
    SELECT * FROM read_csv('{ruta}', header = true, dateformat = '%d/%m/%Y',
                           columns = {{'id_venta': 'TEXT', 'fecha': 'DATE', 'canal': 'TEXT',
                                      'categoria': 'TEXT', 'monto': 'DOUBLE', 'vendedor': 'TEXT'}})
""")

## 1. Entender el archivo

In [ ]:
duckdb.sql("SELECT * FROM ventas LIMIT 8").df()

In [ ]:
duckdb.sql("""
    SELECT COUNT(*) AS filas, MIN(fecha) AS fecha_desde, MAX(fecha) AS fecha_hasta
    FROM ventas
""").df()

## 2. Revisar los montos faltantes

No los completamos con 0: `SUM(monto)` ya ignora los NULL.

In [ ]:
duckdb.sql("""
    SELECT
        COUNT(*) FILTER (WHERE monto IS NULL) AS ventas_sin_monto,
        COUNT(*) AS total_ventas,
        ROUND(100.0 * COUNT(*) FILTER (WHERE monto IS NULL) / COUNT(*), 1) AS porcentaje_faltante
    FROM ventas
""").df()

## 3. Ventas por mes

In [ ]:
duckdb.sql("""
    SELECT
        strftime(fecha, '%Y-%m') AS mes,
        SUM(monto) AS monto_total,
        COUNT(*) AS cantidad_ventas,
        COUNT(*) FILTER (WHERE monto IS NULL) AS ventas_sin_monto_en_el_mes
    FROM ventas
    GROUP BY mes
    ORDER BY mes
""").df()

## 4. Mes de mayor venta

In [ ]:
duckdb.sql("""
    SELECT strftime(fecha, '%Y-%m') AS mes, SUM(monto) AS monto_total
    FROM ventas
    GROUP BY mes
    ORDER BY monto_total DESC
    LIMIT 1
""").df()

## 5. Chequeo de períodos completos

Para no comparar un mes completo contra uno que recién empezó.

In [ ]:
duckdb.sql("""
    SELECT strftime(fecha, '%Y-%m') AS mes, MAX(day(fecha)) AS ultimo_dia_con_datos
    FROM ventas
    GROUP BY mes
    ORDER BY mes
""").df()